# Проект: создание рекомендательной системы для сервиса Яндекс Музыка

Цель проекта - построить прототип персональной рекомендательной системы для музыкального стримингового сервиса на основе истории взаимодействий пользователей с треками.

# Описание данных

Используются три исходных набора данных:

- `tracks.parquet` - информация об 1 млн музыкальных треков, их альбомах, исполнителях и жанрах;
- `catalog_names.parquet` - названия треков, альбомов, исполнителей и жанров;
- `interactions.parquet` - история прослушиваний пользователей.

# Инициализация

Загружаем библиотеки, необходимые для выполнения кода ноутбука.

In [1]:
# импортируем стандартные библиотеки для работы с путями и управления памятью
from pathlib import Path
import gc

# импортируем библиотеки для обработки табличных данных
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# импортируем библиотеки для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# также сразу настраиваем отображение таблиц в ноутбуке
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

Необходимые библиотеки для загрузки, обработки и визуального анализа данных успешно импортированы. Дополнительно подключены инструменты для работы с parquet-файлами и контроля памяти, что важно из-за большого объёма данных о взаимодействиях пользователей. Переходим к загрузке и ознакомлению с данными.

# === ЭТАП 1 ===

# Загрузка первичных данных

Загружаем первичные данные из файлов:
- tracks.parquet
- catalog_names.parquet
- interactions.parquet

На этом этапе загружаем исходные данные и сразу учитываем их объём. Таблицы `tracks` и `catalog_names` читаем полностью, так как они относительно компактны и нужны для проверки связей между треками и каталожными сущностями. Файл `interactions.parquet` значительно крупнее, поэтому пока не загружаем его целиком в память, а работаем с его метаданными через PyArrow.

In [3]:
# задаём пути к исходным файлам с данными
tracks_path = Path("tracks.parquet")
catalog_names_path = Path("catalog_names.parquet")
interactions_path = Path("interactions.parquet")

In [4]:
# загружаем таблицы с информацией о треках и каталогах
tracks = pd.read_parquet(tracks_path)
catalog_names = pd.read_parquet(catalog_names_path)

In [5]:
# создаём объект для работы с большим файлом взаимодействий без полной загрузки в память
interactions_parquet = pq.ParquetFile(interactions_path)

In [6]:
# выводим размеры загруженных данных
print("tracks:", tracks.shape)
print("catalog_names:", catalog_names.shape)
print("interactions:",
    (
        interactions_parquet.metadata.num_rows,
        interactions_parquet.metadata.num_columns,
    ),
)

tracks: (1000000, 4)
catalog_names: (1812471, 3)
interactions: (222629898, 5)


Таблицы `tracks` и `catalog_names` успешно загружены полностью, а для `interactions` получена информация о размере без полной загрузки в память. Набор взаимодействий содержит 222 629 898 строк и 5 столбцов, поэтому дальнейшую работу с ним будем организовывать аккуратно, чтобы не создавать лишнюю нагрузку на память.

# Обзор данных

Проверим структуру и типы данных в исходных таблицах. Особое внимание уделим идентификаторам пользователей, треков, альбомов, исполнителей и жанров, чтобы понять, требуется ли их преобразование перед дальнейшей обработкой.

In [7]:
# выводим типы данных в таблицах с треками и каталожными сущностями
print("tracks:")
print(tracks.dtypes)
print("\ncatalog_names:")
print(catalog_names.dtypes)

tracks:
track_id     int64
albums      object
artists     object
genres      object
dtype: object

catalog_names:
id       int64
type    object
name    object
dtype: object


In [8]:
# выводим схему большого файла взаимодействий без полной загрузки в память
print("interactions:")
print(interactions_parquet.schema_arrow)

interactions:
user_id: int32
track_id: int32
track_seq: int16
started_at: timestamp[ns]
__index_level_0__: int64
-- schema metadata --
pandas: '{"index_columns": ["__index_level_0__"], "column_indexes": [{"na' + 797


Также посмотрим на первые строки в таблицах.

In [9]:
# выводим первые строки таблицы с треками
display(tracks.head())

,track_id,albums,artists,genres
0,26,"[3, 2490753]",[16],"[11, 21]"
1,38,"[3, 2490753]",[16],"[11, 21]"
2,135,"[12, 214, 2490809]",[84],[11]
3,136,"[12, 214, 2490809]",[84],[11]
4,138,"[12, 214, 322, 72275, 72292, 91199, 213505, 2490809, 6007655, 17294156]",[84],[11]


In [10]:
# выводим первые строки таблицы с названиями каталожных сущностей
display(catalog_names.head())

,id,type,name
0,3,album,Taller Children
1,12,album,Wild Young Hearts
2,13,album,Lonesome Crow
3,17,album,Graffiti Soul
4,26,album,Blues Six Pack


In [11]:
# читаем небольшой фрагмент взаимодействий без полной загрузки файла в память
interactions_sample = interactions_parquet.read_row_group(0).to_pandas().head()

# выводим первые строки таблицы взаимодействий
display(interactions_sample)

,user_id,track_id,track_seq,started_at
0,0,99262,1,2022-07-17
1,0,589498,2,2022-07-19
2,0,590262,3,2022-07-21
3,0,590303,4,2022-07-22
4,0,590692,5,2022-07-22


В таблице `tracks` каждому треку соответствуют списки идентификаторов альбомов, исполнителей и жанров. Сами идентификаторы треков хранятся в типе `int64`. Таблица `catalog_names` содержит идентификатор сущности, её тип и название, а `interactions` последовательность прослушиваний пользователей с идентификаторами пользователей и треков, порядковым номером взаимодействия и датой начала прослушивания. В `interactions` идентификаторы уже имеют компактный тип `int32`, тогда как в `tracks` и `catalog_names` используются `int64`, поэтому следующим шагом проверим диапазон значений и решим, требуется ли преобразование типов. Для этого проверим минимальные и максимальные значения идентификаторов в таблицах `tracks` и `catalog_names`. Это позволит определить, укладываются ли значения в диапазон `int32` и можно ли привести идентификаторы к более компактному типу без потери данных.

In [12]:
# проверяем диапазон идентификаторов треков
print("tracks.track_id:")
print("min:", tracks["track_id"].min())
print("max:", tracks["track_id"].max())

tracks.track_id:
min: 26
max: 101521819


In [13]:
# проверяем диапазон идентификаторов каталожных сущностей
print("\ncatalog_names.id:")
print("min:", catalog_names["id"].min())
print("max:", catalog_names["id"].max())


catalog_names.id:
min: 0
max: 101521819


In [14]:
# выводим допустимый диапазон типа int32
int32_info = np.iinfo(np.int32)
print("\nint32 range:")
print("min:", int32_info.min)
print("max:", int32_info.max)


int32 range:
min: -2147483648
max: 2147483647


Максимальные значения идентификаторов в `tracks` и `catalog_names` значительно меньше верхней границы типа `int32`. Поэтому `track_id` и `id` можно безопасно привести из `int64` к `int32` без потери данных. Это также уменьшит потребление памяти и приведёт типы идентификаторов к формату, уже используемому в `interactions`.

In [15]:
# приводим идентификатор трека к более компактному типу
tracks["track_id"] = tracks["track_id"].astype("int32")

In [16]:
# приводим идентификатор каталожной сущности к более компактному типу
catalog_names["id"] = catalog_names["id"].astype("int32")

In [17]:
# делаем проверку
print("tracks.track_id:", tracks["track_id"].dtype)
print("catalog_names.id:", catalog_names["id"].dtype)

tracks.track_id: int32
catalog_names.id: int32


Идентификаторы `track_id` и `id` успешно приведены к типу `int32`. Теперь типы идентификаторов согласованы между таблицами, а хранение данных стало более компактным без потери информации.

Далее проверим состав каталога и убедимся, какие типы сущностей представлены в `catalog_names`. Это необходимо перед проверкой соответствия альбомов, исполнителей и жанров из `tracks` справочнику каталога.

In [18]:
# выводим уникальные типы каталожных сущностей
print(catalog_names["type"].value_counts())

type
track     1000000
album      658724
artist     153581
genre         166
Name: count, dtype: int64


В `catalog_names` представлены все основные типы сущностей, используемые в данных: треки, альбомы, исполнители и жанры. Это позволяет проверить, все ли идентификаторы альбомов, исполнителей и жанров из таблицы `tracks` имеют соответствующие записи в каталоге. Теперь сравним идентификаторы альбомов, исполнителей и жанров из `tracks` с соответствующими идентификаторами в `catalog_names`. Идентификаторы, отсутствующие в каталоге, будем считать неизвестными.

In [19]:
# формируем множества идентификаторов сущностей из каталога
catalog_album_ids = set(catalog_names.loc[catalog_names["type"] == "album", "id"])
catalog_artist_ids = set(catalog_names.loc[catalog_names["type"] == "artist", "id"])
catalog_genre_ids = set(catalog_names.loc[catalog_names["type"] == "genre", "id"])

In [20]:
# собираем уникальные идентификаторы сущностей, используемые в треках
track_album_ids = {item_id for ids in tracks["albums"] for item_id in ids}
track_artist_ids = {item_id for ids in tracks["artists"] for item_id in ids}
track_genre_ids = {item_id for ids in tracks["genres"] for item_id in ids}

In [21]:
# находим идентификаторы, отсутствующие в каталоге
unknown_album_ids = track_album_ids - catalog_album_ids
unknown_artist_ids = track_artist_ids - catalog_artist_ids
unknown_genre_ids = track_genre_ids - catalog_genre_ids

In [22]:
# выводим количество неизвестных сущностей
print("неизвестные альбомы:", len(unknown_album_ids))
print("неизвестные исполнители:", len(unknown_artist_ids))
print("неизвестные жанры:", len(unknown_genre_ids))

неизвестные альбомы: 0
неизвестные исполнители: 0
неизвестные жанры: 30


Неизвестных альбомов и исполнителей не обнаружено: все их идентификаторы присутствуют в `catalog_names`. Для жанров найдено 30 идентификаторов, отсутствующих в каталоге, поэтому следующим шагом проверим, какие именно это значения и насколько часто они встречаются в треках.

In [23]:
# выводим идентификаторы неизвестных жанров
print("unknown genre ids:")
print(sorted(unknown_genre_ids))

unknown genre ids:
[124, 126, 130, 131, 132, 133, 134, 135, 146, 148, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169]


In [24]:
# считаем, сколько треков содержат каждый неизвестный жанр
unknown_genre_counts = {}
for genre_ids in tracks["genres"]:
    for genre_id in genre_ids:
        if genre_id in unknown_genre_ids:
            unknown_genre_counts[genre_id] = unknown_genre_counts.get(genre_id, 0) + 1

In [25]:
# выводим частоту встречаемости неизвестных жанров
print(
    sorted(
        unknown_genre_counts.items(),
        key=lambda x: x[1],
        reverse=True
    )
)

[(154, 8170), (161, 6750), (163, 5351), (158, 4867), (157, 3994), (151, 3387), (169, 2780), (155, 2288), (159, 2087), (164, 1961), (156, 1560), (162, 962), (168, 789), (153, 704), (160, 601), (165, 409), (146, 338), (167, 336), (131, 256), (152, 245), (126, 136), (130, 107), (132, 106), (134, 91), (135, 39), (133, 27), (166, 20), (148, 4), (124, 2), (150, 2)]


Неизвестные жанры встречаются с разной частотой: некоторые из них указаны у нескольких тысяч треков. Поэтому перед корректировкой данных проверим, сколько треков затронуто и есть ли среди них треки, для которых все жанры являются неизвестными.

In [26]:
# определяем треки, содержащие хотя бы один неизвестный жанр
has_unknown_genre = tracks["genres"].apply(lambda genre_ids: any(genre_id in unknown_genre_ids for genre_id in genre_ids))

In [27]:
# определяем треки, у которых нет ни одного известного жанра
has_only_unknown_genres = tracks["genres"].apply(
    lambda genre_ids: (
        len(genre_ids) > 0
        and all(genre_id in unknown_genre_ids for genre_id in genre_ids)
    )
)

In [28]:
# выводим количество затронутых треков
print("Треков с неизвестными жанрами:", has_unknown_genre.sum())
print("Треков только с неизвестными жанрами:", has_only_unknown_genres.sum())

Треков с неизвестными жанрами: 48345
Треков только с неизвестными жанрами: 7


Неизвестные жанры встречаются у 48345 треков, однако только у 7 треков все указанные жанры отсутствуют в каталоге. Следовательно, для подавляющего большинства затронутых треков можно удалить неизвестные идентификаторы, сохранив при этом корректную жанровую информацию. Но для начала проверим, есть ли в исходных данных треки с пустым списком жанров. Это поможет решить, как корректно обработать данные 7 треков.

In [29]:
# проверяем количество треков без указанных жанров
tracks_without_genres = tracks["genres"].apply(len).eq(0).sum()
print("Треков без указанных жанров:", tracks_without_genres)

Треков без указанных жанров: 3687


В исходных данных уже присутствуют 3687 треков без указанных жанров. Поэтому для 7 треков, содержащих только неизвестные жанры, после удаления некорректных идентификаторов допустимо оставить пустой список жанров, не вводя искусственную категорию.

In [30]:
# удаляем неизвестные идентификаторы жанров, сохраняя все известные жанры
tracks["genres"] = tracks["genres"].apply(
    lambda genre_ids: [
        genre_id
        for genre_id in genre_ids
        if genre_id not in unknown_genre_ids
    ]
)

In [31]:
# повторно проверяем наличие неизвестных жанров после очистки
remaining_unknown_genres = {
    genre_id
    for genre_ids in tracks["genres"]
    for genre_id in genre_ids
    if genre_id not in catalog_genre_ids
}

In [32]:
# проверяем количество треков без жанров после очистки и выводим результат
tracks_without_genres_after = tracks["genres"].apply(len).eq(0).sum()
print("Неизвестных жанров после очистки:", len(remaining_unknown_genres))
print("Треков без указанных жанров после очистки:", tracks_without_genres_after)

Неизвестных жанров после очистки: 0
Треков без указанных жанров после очистки: 3694


Неизвестные идентификаторы жанров успешно удалены: после очистки их количество равно нулю. Число треков без жанров увеличилось с 3687 до 3694, то есть ровно на 7 треков, у которых до очистки были указаны только неизвестные жанры.

Проведём финальную проверку соответствия альбомов, исполнителей и жанров каталогу после очистки. Это позволит убедиться, что в данных больше не осталось неизвестных идентификаторов сущностей.

In [33]:
# повторно собираем уникальные идентификаторы сущностей из треков
track_album_ids = {item_id for ids in tracks["albums"] for item_id in ids}
track_artist_ids = {item_id for ids in tracks["artists"] for item_id in ids}
track_genre_ids = {item_id for ids in tracks["genres"] for item_id in ids}

In [34]:
# повторно проверяем неизвестные идентификаторы
unknown_album_ids = track_album_ids - catalog_album_ids
unknown_artist_ids = track_artist_ids - catalog_artist_ids
unknown_genre_ids = track_genre_ids - catalog_genre_ids

In [35]:
# выводим результаты контрольной проверки
print("Неизвестных альбомов:", len(unknown_album_ids))
print("Неизвестных исполнителей:", len(unknown_artist_ids))
print("Неизвестных жанров:", len(unknown_genre_ids))

Неизвестных альбомов: 0
Неизвестных исполнителей: 0
Неизвестных жанров: 0


После очистки неизвестных сущностей не осталось: все идентификаторы альбомов, исполнителей и жанров из `tracks` присутствуют в `catalog_names`. Таким образом, проблема несогласованности справочника и данных устранена.

# Выводы

- На этапе первичного знакомства с данными были проверены: структура таблиц, типы идентификаторов и согласованность связей между треками и сущностями в каталогах.
- Идентификаторы `track_id` в `tracks` и `id` в `catalog_names` изначально имели тип `int64`. Проверка диапазона значений показала, что они полностью укладываются в `int32`, поэтому были безопасно приведены к более компактному типу без потери данных. В `interactions` идентификаторы уже хранятся в `int32`.
- Для альбомов и исполнителей неизвестных идентификаторов не обнаружено. Для жанров было найдено 30 идентификаторов, отсутствующих в `catalog_names`. Они встречались у 48345 треков, причём только у 7 треков все жанры были неизвестными. Неизвестные жанровые идентификаторы были удалены, при этом корректные жанры сохранены. После очистки неизвестных альбомов, исполнителей и жанров не осталось.

Таким образом, основные проблемы первичных данных были выявлены и устранены, а данные подготовлены к дальнейшему исследовательскому анализу (EDA).

# === ЭТАП 2 ===

# EDA

Распределение количества прослушанных треков.

Наиболее популярные треки

Наиболее популярные жанры

Треки, которые никто не прослушал

# Преобразование данных

Преобразуем данные в формат, более пригодный для дальнейшего использования в расчётах рекомендаций.

# Сохранение данных

Сохраним данные в двух файлах в персональном S3-бакете по пути `recsys/data/`:
- `items.parquet` — все данные о музыкальных треках,
- `events.parquet` — все данные о взаимодействиях.

# Очистка памяти

Здесь, может понадобится очистка памяти для высвобождения ресурсов для выполнения кода ниже. 

Приведите соответствующие код, комментарии, например:
- код для удаление более ненужных переменных,
- комментарий, что следует перезапустить kernel, выполнить такие-то начальные секции и продолжить с этапа 3.

# === ЭТАП 3 ===

# Загрузка данных

Если необходимо, то загружаем items.parquet, events.parquet.

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

# Топ популярных

Рассчитаем рекомендации как топ популярных.

# Персональные

Рассчитаем персональные рекомендации.

# Похожие

Рассчитаем похожие, они позже пригодятся для онлайн-рекомендаций.

# Построение признаков

Построим три признака, можно больше, для ранжирующей модели.

# Ранжирование рекомендаций

Построим ранжирующую модель, чтобы сделать рекомендации более точными. Отранжируем рекомендации.

# Оценка качества

Проверим оценку качества трёх типов рекомендаций: 

- топ популярных,
- персональных, полученных при помощи ALS,
- итоговых
  
по четырем метрикам: recall, precision, coverage, novelty.

# === Выводы, метрики ===

Основные выводы при работе над расчётом рекомендаций, рассчитанные метрики.